# 02 — Finance & Safety Agents
**Agentic X | Studio 67**

### Overview
This notebook evaluates the **Finance** and **Safety** agents using the validated Studio 67 dataset (**150 test cases, 450 solution options**). Both agents use deterministic Python rules and provide structured **Accepted/Rejected** decisions and reasons.

### Agent Roles
- **Finance:** compares projected spending (including costs, existing commitments, and savings as applicable) with the **current budget**.
- **Safety:** checks recorded safety compliance, permits, verified mitigation, and residual risk (acceptable **≤ 3**). A failed check triggers a safety veto.

### Related Python Files
- `agents/finance.py` — Finance decisions and calculations.
- `agents/safety.py` — Safety checks and veto decisions.
- `core/environment.py` — Reads and validates scenario and option data without sending ground-truth evaluation columns to agents.
- `core/models.py` — Shared structured agent outputs.

### Requirements & Setup
Python 3.10+, `pandas`, `openpyxl`, `pydantic` (installed from the repository's `requirements.txt`). **No API key is required** for these two rule-based agents. Run cells in order from the repository root; update the two `.py` files in GitHub before running this notebook. The notebook never overwrites agent source files.

**Project requirement covered:** agent responsibilities, PEAS/agent types, environment variables, and testing in multiple realistic scenarios. Negotiation, ACL, and live UI are addressed in subsequent notebooks.

### Cell 1 — Connect to Project and Install Requirements

In [1]:
from pathlib import Path
import sys, os, subprocess

REPO_URL = "https://github.com/ProjectsSpaceHub/AgenticX.git"
ROOT = Path("/content/AgenticX")
if not (ROOT / "core" / "environment.py").exists():
    if ROOT.exists():
        raise RuntimeError("Project directory exists but its core files are missing. Check ROOT.")
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Project root:", ROOT)
print("Setup complete. Existing clones are not automatically updated; run git pull if needed.")

Project root: /content/AgenticX
Setup complete. Existing clones are not automatically updated; run git pull if needed.


### Cell 2 — Import Libraries and Verify Updated Agents

In [2]:
import importlib
import pandas as pd
from IPython.display import display
from core.environment import load_dataset, get_case, clean, HIDDEN_CASE_FIELDS, HIDDEN_OPTION_FIELDS
import agents.finance as finance_module
import agents.safety as safety_module

importlib.reload(finance_module)
importlib.reload(safety_module)
assert getattr(finance_module.FinanceAgent, "VERSION", None) == "2026-11-finance-v2", "Update agents/finance.py in the local repo first"
assert getattr(safety_module.SafetyAgent, "VERSION", None) == "2026-11-safety-v2", "Update agents/safety.py in the local repo first"
finance_agent = finance_module.FinanceAgent()
safety_agent = safety_module.SafetyAgent()
print("Finance and Safety: updated rule-based source verified; no LLM key needed.")

Finance and Safety: updated rule-based source verified; no LLM key needed.


### Cell 3 — Load and Validate Dataset

In [3]:
cases_df, options_df = load_dataset()
assert len(cases_df) == 150 and len(options_df) == 450
assert cases_df["case_id"].nunique() == 150
assert options_df.groupby("case_id").size().eq(3).all()
print("Test cases:", len(cases_df))
print("Solution options:", len(options_df))
display(cases_df[["case_id", "scenario", "incident", "initial_budget_sar", "time_left_min"]].head())
display(options_df[["case_id", "option_id", "option_name", "cost_sar", "quality_score"]].head())

Test cases: 150
Solution options: 450


,case_id,scenario,incident,initial_budget_sar,time_left_min
0,SCR-001,Screen Failure,Main LED screen failed during final rehearsal,8500,100
1,SCR-002,Screen Failure,LED wall lost signal before keynote,2400,60
2,SCR-003,Screen Failure,Display controller failure before exhibition o...,7220,90
3,SCR-004,Screen Failure,Screen power module stopped working,14000,150
4,SCR-005,Screen Failure,Main stage screen shows intermittent blackout,11000,150


,case_id,option_id,option_name,cost_sar,quality_score
0,SCR-001,SCR-001-O1,Use backup LED,1290,89
1,SCR-001,SCR-001-O2,Rent replacement screen,5920,89
2,SCR-001,SCR-001-O3,Continue with projection,1210,60
3,SCR-002,SCR-002-O1,Use backup LED,1230,75
4,SCR-002,SCR-002-O2,Rent replacement screen,6300,96


### Cell 4 — Finance and Safety PEAS & Agent Types

In [4]:
peas_df = pd.DataFrame([
    {"Agent":"Finance", "Performance Measure":"Respect current budget; avoid excessive total spending",
     "Environment":"Studio 67 event-crisis simulation", "Actuators":"Approve/Refuse and explain financial assessment",
     "Sensors":"Current budget, committed/projected spending, option cost, gross savings",
     "Type":"Deliberative (deterministic calculation)",
     "Justification":"Computes financial consequences of each option before choosing an action"},
    {"Agent":"Safety", "Performance Measure":"Prevent unsafe approvals and permit violations",
     "Environment":"Studio 67 event-crisis simulation", "Actuators":"Approve/Refuse and issue safety veto",
     "Sensors":"Safety OK, permit OK, mitigation verification, residual risk",
     "Type":"Reactive (rule-based)",
     "Justification":"Immediately vetoes an option when any mandatory recorded check fails"},
])
display(peas_df)

,Agent,Performance Measure,Environment,Actuators,Sensors,Type,Justification
0,Finance,Respect current budget; avoid excessive total ...,Studio 67 event-crisis simulation,Approve/Refuse and explain financial assessment,"Current budget, committed/projected spending, ...",Deliberative (deterministic calculation),Computes financial consequences of each option...
1,Safety,Prevent unsafe approvals and permit violations,Studio 67 event-crisis simulation,Approve/Refuse and issue safety veto,"Safety OK, permit OK, mitigation verification,...",Reactive (rule-based),Immediately vetoes an option when any mandator...


### Cell 5 — Evaluate Screen Failure (SCR-001)

In [5]:
case, options = get_case("SCR-001")
results = []
for option in options:
    for agent in (finance_agent, safety_agent):
        e = agent.evaluate(case, option)
        results.append({"case_id": case["case_id"], **e.model_dump()})
results_df = pd.DataFrame(results)
assert len(results_df) == 6
assert set(results_df.agent) == {"Finance", "Safety"}
display(results_df[["case_id", "option_id", "agent", "approved", "reason"]])

,case_id,option_id,agent,approved,reason
0,SCR-001,SCR-001-O1,Finance,True,"Committed spending 0.00 SAR + option cost 1,29..."
1,SCR-001,SCR-001-O1,Safety,True,Safety OK: True; Permit OK: True; Mitigation v...
2,SCR-001,SCR-001-O2,Finance,True,"Committed spending 0.00 SAR + option cost 5,92..."
3,SCR-001,SCR-001-O2,Safety,True,Safety OK: True; Permit OK: True; Mitigation v...
4,SCR-001,SCR-001-O3,Finance,True,"Committed spending 0.00 SAR + option cost 1,21..."
5,SCR-001,SCR-001-O3,Safety,True,Safety OK: True; Permit OK: True; Mitigation v...


### Cell 6 — Evaluate Budget Overrun & Safety Conflict

In [6]:
scenario_evaluations = []
for case_id in ("BDG-001", "SFT-001"):
    test_case, test_options = get_case(case_id)
    for option in test_options:
        fin = finance_agent.evaluate(test_case, option)
        saf = safety_agent.evaluate(test_case, option)
        scenario_evaluations.append({
            "case_id": case_id, "option_id": option["option_id"],
            "finance_approved": fin.approved, "finance_reason": fin.reason,
            "safety_approved": saf.approved, "safety_reason": saf.reason,
        })
scenario_df = pd.DataFrame(scenario_evaluations)
assert len(scenario_df) == 6
display(scenario_df)

,case_id,option_id,finance_approved,finance_reason,safety_approved,safety_reason
0,BDG-001,BDG-001-O1,True,"Projected spending before action 22,286.00 SAR...",True,Safety OK: True; Permit OK: True; Mitigation v...
1,BDG-001,BDG-001-O2,True,"Projected spending before action 22,286.00 SAR...",True,Safety OK: True; Permit OK: True; Mitigation v...
2,BDG-001,BDG-001-O3,False,"Projected spending before action 22,286.00 SAR...",True,Safety OK: True; Permit OK: True; Mitigation v...
3,SFT-001,SFT-001-O1,True,Committed spending 0.00 SAR + option cost 350....,False,Safety OK: False; Permit OK: False; Mitigation...
4,SFT-001,SFT-001-O2,True,"Committed spending 0.00 SAR + option cost 1,81...",True,Safety OK: True; Permit OK: True; Mitigation v...
5,SFT-001,SFT-001-O3,True,"Committed spending 0.00 SAR + option cost 3,14...",True,Safety OK: True; Permit OK: True; Mitigation v...


### Cell 7 — Test Finance Calculations & Budget Overrides

In [7]:
# Budget Overrun: projected + cost - gross savings; boundaries inclusive.
base = {"scenario":"Budget Overrun", "initial_budget_sar":10000, "projected_total_spend_sar":12000}
option = {"option_id":"FIN-TEST", "cost_sar":200, "gross_savings_sar":2500}
assert finance_agent.evaluate(base, option).approved is True  # 9700 <= 10000
assert finance_agent.evaluate({**base,"initial_budget_sar":9000}, option).approved is False

# Zero must never be replaced with fallback budget_sar.
zero_case = {"scenario":"Screen Failure", "initial_budget_sar":0,
             "budget_sar":100000, "committed_spend_sar":0}
zero_option = {"option_id":"FIN-ZERO", "cost_sar":1}
assert finance_agent.evaluate(zero_case, zero_option).approved is False
assert finance_agent.evaluate(zero_case, {**zero_option, "cost_sar":0}).approved is True

# Include existing commitments, not only option cost.
committed_case = {"scenario":"Screen Failure", "initial_budget_sar":1000, "committed_spend_sar":800}
assert finance_agent.evaluate(committed_case, {"option_id":"FIN-C", "cost_sar":250}).approved is False
assert finance_agent.evaluate(committed_case, {"option_id":"FIN-C", "cost_sar":200}).approved is True

# Required financial inputs must not be silently substituted with zero.
for invalid in (None, float("nan"), -10, float("inf")):
    try:
        finance_agent.evaluate({**base, "initial_budget_sar":invalid}, option)
        raise AssertionError(f"Budget {invalid!r} was incorrectly accepted")
    except ValueError:
        pass
print("Finance budget, savings, override, and validation tests passed!")

Finance budget, savings, override, and validation tests passed!


### Cell 8 — Test Safety Hard Veto & Risk Boundaries

In [8]:
safe = {"option_id":"SAFE-TEST", "safety_ok":True, "permit_ok":True,
        "mitigation_verified":True, "risk_score_after_mitigation":3}
assert safety_agent.evaluate({}, safe).approved is True
assert safety_agent.evaluate({}, {**safe,"risk_score_after_mitigation":0}).approved is True
assert safety_agent.evaluate({}, {**safe,"risk_score_after_mitigation":4}).approved is False
for field in ("safety_ok", "permit_ok", "mitigation_verified"):
    assert safety_agent.evaluate({}, {**safe,field:False}).approved is False
for invalid in (None, "unknown", float("nan"), float("inf"), -1):
    evaluation = safety_agent.evaluate({}, {**safe,"risk_score_after_mitigation":invalid})
    assert evaluation.approved is False, f"Invalid risk {invalid!r} incorrectly approved"
assert not safety_agent.evaluate({}, {"option_id":"SAFE-MISSING"}).approved
print("Safety hard veto and risk boundary tests passed!")

Safety hard veto and risk boundary tests passed!


### Cell 9 — Check Decision and Explanation Consistency

In [9]:
# A decision must agree with its own explanation.
checks = [
    finance_agent.evaluate({"scenario":"Screen Failure","initial_budget_sar":1000,"committed_spend_sar":0},
                           {"option_id":"F-X","cost_sar":1290}),
    finance_agent.evaluate({"scenario":"Screen Failure","initial_budget_sar":1500,"committed_spend_sar":0},
                           {"option_id":"F-X","cost_sar":1290}),
    safety_agent.evaluate({}, safe),
    safety_agent.evaluate({}, {**safe,"permit_ok":False}),
]
for evaluation in checks:
    if evaluation.approved:
        assert "Accepted" in evaluation.reason and "Rejected" not in evaluation.reason
    else:
        assert "Rejected" in evaluation.reason and "Accepted" not in evaluation.reason
    assert evaluation.blocking is True
    print(evaluation.agent, evaluation.approved, "|", evaluation.reason)
print("Decision/explanation consistency verified!")

Finance False | Committed spending 0.00 SAR + option cost 1,290.00 SAR = projected total 1,290.00 SAR. Current budget: 1,000.00 SAR; above the current budget by 290.00 SAR; Rejected.
Finance True | Committed spending 0.00 SAR + option cost 1,290.00 SAR = projected total 1,290.00 SAR. Current budget: 1,500.00 SAR; within the current budget by 210.00 SAR; Accepted.
Safety True | Safety OK: True; Permit OK: True; Mitigation verified: True; Residual risk: 3 (must be <= 3). All recorded safety checks passed; Accepted.
Safety False | Safety OK: True; Permit OK: False; Mitigation verified: True; Residual risk: 3 (must be <= 3). Safety veto (permit_ok); Rejected.
Decision/explanation consistency verified!


### Cell 10 — Verify Ground-Truth Separation

In [10]:
demo_case, demo_options = get_case("SFT-001")
assert not HIDDEN_CASE_FIELDS.intersection(demo_case)
for demo in demo_options:
    assert not HIDDEN_OPTION_FIELDS.intersection(demo)
print("Evaluation-only columns are excluded from all agent inputs.")

Evaluation-only columns are excluded from all agent inputs.


### Cell 11 — Evaluate All 150 Cases / 450 Solutions

In [11]:
# This uses only Python rules; it makes NO Gemini or other LLM API calls.
options_by_case = {
    key: group.to_dict("records")
    for key, group in options_df.groupby("case_id")
}
rows = []
for raw_case in cases_df.to_dict("records"):
    c = {k:clean(v) for k,v in raw_case.items() if k not in HIDDEN_CASE_FIELDS}
    for raw_option in options_by_case[c["case_id"]]:
        o = {k:clean(v) for k,v in raw_option.items() if k not in HIDDEN_OPTION_FIELDS}
        f = finance_agent.evaluate(c,o)
        s = safety_agent.evaluate(c,o)
        rows.append({
            "case_id":c["case_id"], "scenario":c["scenario"], "option_id":o["option_id"],
            "finance_approved":f.approved, "safety_approved":s.approved,
            "both_approved": f.approved and s.approved,
            "finance_reason":f.reason, "safety_reason":s.reason,
        })
batch_df = pd.DataFrame(rows)
assert len(batch_df) == 450
assert batch_df.case_id.nunique() == 150
assert batch_df.option_id.is_unique
print("Rule evaluations:", len(batch_df), "options across", batch_df.case_id.nunique(), "cases")
print("Finance approvals:", int(batch_df.finance_approved.sum()))
print("Safety approvals:", int(batch_df.safety_approved.sum()))
display(batch_df.head(9))

Rule evaluations: 450 options across 150 cases
Finance approvals: 357
Safety approvals: 355


,case_id,scenario,option_id,finance_approved,safety_approved,both_approved,finance_reason,safety_reason
0,SCR-001,Screen Failure,SCR-001-O1,True,True,True,"Committed spending 0.00 SAR + option cost 1,29...",Safety OK: True; Permit OK: True; Mitigation v...
1,SCR-001,Screen Failure,SCR-001-O2,True,True,True,"Committed spending 0.00 SAR + option cost 5,92...",Safety OK: True; Permit OK: True; Mitigation v...
2,SCR-001,Screen Failure,SCR-001-O3,True,True,True,"Committed spending 0.00 SAR + option cost 1,21...",Safety OK: True; Permit OK: True; Mitigation v...
3,SCR-002,Screen Failure,SCR-002-O1,True,True,True,"Committed spending 0.00 SAR + option cost 1,23...",Safety OK: True; Permit OK: True; Mitigation v...
4,SCR-002,Screen Failure,SCR-002-O2,False,True,False,"Committed spending 0.00 SAR + option cost 6,30...",Safety OK: True; Permit OK: True; Mitigation v...
5,SCR-002,Screen Failure,SCR-002-O3,True,True,True,"Committed spending 0.00 SAR + option cost 1,23...",Safety OK: True; Permit OK: True; Mitigation v...
6,SCR-003,Screen Failure,SCR-003-O1,True,True,True,"Committed spending 0.00 SAR + option cost 1,18...",Safety OK: True; Permit OK: True; Mitigation v...
7,SCR-003,Screen Failure,SCR-003-O2,True,True,True,"Committed spending 0.00 SAR + option cost 6,62...",Safety OK: True; Permit OK: True; Mitigation v...
8,SCR-003,Screen Failure,SCR-003-O3,True,True,True,"Committed spending 0.00 SAR + option cost 1,05...",Safety OK: True; Permit OK: True; Mitigation v...


### Cell 12 — Performance Summary

In [12]:
summary_df = batch_df.groupby("scenario").agg(
    total_options=("option_id","count"),
    finance_approved=("finance_approved","sum"),
    safety_approved=("safety_approved","sum"),
    both_approved=("both_approved","sum"),
).reset_index()
for name in ("finance_approved","safety_approved","both_approved"):
    summary_df[name+"_rate_pct"] = (100 * summary_df[name] / summary_df.total_options).round(2)
display(summary_df)
print("These are deterministic option-level rule approval rates, NOT full multi-agent accuracy.")

,scenario,total_options,finance_approved,safety_approved,both_approved,finance_approved_rate_pct,safety_approved_rate_pct,both_approved_rate_pct
0,Budget Overrun,150,100,133,91,66.67,88.67,60.67
1,Safety Conflict,150,130,92,77,86.67,61.33,51.33
2,Screen Failure,150,127,130,109,84.67,86.67,72.67


These are deterministic option-level rule approval rates, NOT full multi-agent accuracy.


### Cell 13 — Save Evaluation Results

In [13]:
results_dir = ROOT / "results"
results_dir.mkdir(exist_ok=True)
batch_df.to_csv(results_dir / "finance_safety_evaluations.csv", index=False)
summary_df.to_csv(results_dir / "finance_safety_summary.csv", index=False)
print("Saved:", results_dir / "finance_safety_evaluations.csv")
print("Saved:", results_dir / "finance_safety_summary.csv")

Saved: /content/AgenticX/results/finance_safety_evaluations.csv
Saved: /content/AgenticX/results/finance_safety_summary.csv


### Cell 14 — Verify Python Source Paths

In [14]:
import inspect
from core.models import AgentEvaluation
print("Finance:", inspect.getfile(finance_module.FinanceAgent))
print("Safety:", inspect.getfile(safety_module.SafetyAgent))
assert Path(inspect.getfile(finance_module.FinanceAgent)).resolve() == (ROOT / "agents/finance.py").resolve()
assert Path(inspect.getfile(safety_module.SafetyAgent)).resolve() == (ROOT / "agents/safety.py").resolve()
assert finance_module.FinanceAgent.VERSION == "2026-11-finance-v2"
assert safety_module.SafetyAgent.VERSION == "2026-11-safety-v2"
print("The notebook uses the updated repository agent modules.")

Finance: /content/AgenticX/agents/finance.py
Safety: /content/AgenticX/agents/safety.py
The notebook uses the updated repository agent modules.


### Conclusion
Finance and Safety were evaluated with deterministic Python logic on all **150 synthetic cases and 450 options**. Finance compares computed total spending with the current scenario budget; Safety enforces four mandatory safety conditions including residual risk ≤ 3. Both return structured, auditable approvals/refusals. These are **individual rule-check results**, not evidence that the full four-agent negotiation succeeded. That integration is verified in later notebooks.